In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1un2SaLv7_DvXerUlPtzKSNZ_j-mu1qFQ/"
    "EEG_GANet_Reproduction"
)

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(NOTEBOOK_DIR)

print("Notebook directory:", NOTEBOOK_DIR)
print("Directory exists:", NOTEBOOK_DIR.is_dir())
print("Current directory:", Path.cwd())

In [ ]:
# ============================================================
# STEP 1 — DRIVE CONNECTION AND EXISTING-FILE INVENTORY
#
# Notebook:
# EEG_GANet_Reproduction/notebooks/DBNet_GLASS_Pilot.ipynb
#
# Output directory:
# EEG_GANet_Reproduction/GLASS_GANet/results/
# multisubject_replication/dbnet_glass_structured_pilot/
#
# CPU runtime is sufficient for this step.
# ============================================================

import json
import os
import tempfile
import uuid
import zipfile

from datetime import datetime, timezone
from pathlib import Path

import numpy as np


PROJECT_NAME = "EEG_GANet_Reproduction"

SHARED_TARGET_ID = (
    "1un2SaLv7_DvXerUlPtzKSNZ_j-mu1qFQ"
)

SUBJECTS = [
    f"A{i:02d}"
    for i in range(1, 9)
]


# ------------------------------------------------------------
# 1. Safe filesystem helpers
# ------------------------------------------------------------

def readable_directory(path):
    try:
        with os.scandir(path) as entries:
            next(entries, None)
        return True
    except OSError:
        return False


def child_directories(path):
    try:
        return [
            p
            for p in Path(path).iterdir()
            if readable_directory(p)
        ]
    except OSError:
        return []


def safe_is_file(path):
    try:
        return Path(path).is_file()
    except OSError:
        return False


def archive_headers(path):
    """
    Inspect NumPy array names, shapes and dtypes.

    Does not load prediction values or unpickle objects.
    """
    headers = {}

    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            if not name.endswith(".npy"):
                continue

            with archive.open(name) as member:
                version = np.lib.format.read_magic(member)

                if version == (1, 0):
                    shape, order, dtype = (
                        np.lib.format.read_array_header_1_0(
                            member
                        )
                    )

                elif version == (2, 0):
                    shape, order, dtype = (
                        np.lib.format.read_array_header_2_0(
                            member
                        )
                    )

                else:
                    headers[name[:-4]] = {
                        "header_version": list(version)
                    }
                    continue

            headers[name[:-4]] = {
                "shape": list(shape),
                "dtype": str(dtype),
                "fortran_order": bool(order),
            }

    return headers


# ------------------------------------------------------------
# 2. Find an accessible Drive mount
# ------------------------------------------------------------

drive_roots = []

previous_drive = globals().get("DRIVE_ROOT")

if previous_drive is not None:
    drive_roots.append(Path(previous_drive))

drive_roots.extend(
    child_directories("/content")
)

drive_roots = list(
    dict.fromkeys(
        root
        for root in drive_roots
        if readable_directory(root / "MyDrive")
    )
)

if not drive_roots:
    from google.colab import drive

    new_mount = Path(
        tempfile.mkdtemp(
            prefix="eeg_drive_",
            dir="/content",
        )
    )

    print(
        "Select the Google account with access "
        "to the shared project.",
        flush=True,
    )

    drive.mount(str(new_mount))

    if not readable_directory(
        new_mount / "MyDrive"
    ):
        raise RuntimeError(
            "The new Drive mount is not readable."
        )

    drive_roots = [new_mount]

print("\nAccessible Drive mounts:")

for root in drive_roots:
    print(root)


# ------------------------------------------------------------
# 3. Recover the project directory
# ------------------------------------------------------------

project_candidates = []

previous_project = globals().get("PROJECT_ROOT")

if previous_project is not None:
    project_candidates.append(
        Path(previous_project)
    )

for root in drive_roots:
    project_candidates.append(
        root
        / ".shortcut-targets-by-id"
        / SHARED_TARGET_ID
        / PROJECT_NAME
    )

    project_candidates.extend(
        path
        for path in child_directories(
            root / "MyDrive"
        )
        if path.name.startswith(PROJECT_NAME)
    )

project_candidates = list(
    dict.fromkeys(project_candidates)
)

# A01's verified file is the canonical GLASS-domain dataset.
# Do not require an A01_processed_8ch file.
canonical_a01_relative = (
    Path("GLASS_GANet")
    / "prepared_data"
    / "A01_GLASSdomain_128_split42_glassseed1.npz"
)

resolved_project = next(
    (
        path
        for path in project_candidates
        if safe_is_file(
            path / canonical_a01_relative
        )
    ),
    None,
)

if resolved_project is None:
    print("\nChecked project paths:")

    for path in project_candidates:
        print(path)

    raise FileNotFoundError(
        "The known A01 canonical dataset was not "
        "found at these paths. This does not establish "
        "that the Google account is wrong. Copy the "
        "project folder's current path from Colab's "
        "Files panel, assign it to PROJECT_ROOT using "
        "Path(...), and rerun this cell."
    )

PROJECT_ROOT = resolved_project

DRIVE_ROOT = next(
    (
        root
        for root in drive_roots
        if root in PROJECT_ROOT.parents
    ),
    None,
)

PREPARED_DIR = (
    PROJECT_ROOT
    / "GLASS_GANet"
    / "prepared_data"
)

RESULTS_ROOT = (
    PROJECT_ROOT
    / "GLASS_GANet"
    / "results"
)

REPLICATION_DIR = (
    RESULTS_ROOT
    / "multisubject_replication"
)

HYBRID_DIR = (
    REPLICATION_DIR
    / "dbnet_glass_structured_pilot"
)

print("\nProject:", PROJECT_ROOT)
print("Prepared data:", PREPARED_DIR)
print("New experiment output:", HYBRID_DIR)


# ------------------------------------------------------------
# 4. Inspect all eight canonical dataset headers
# ------------------------------------------------------------

expected_shapes = {
    "X_train_8ch": [2520, 8, 128],
    "X_validation_8ch": [840, 8, 128],
    "y_train": [2520],
    "y_validation": [840],
}

dataset_records = {}
problems = []

print("\nCanonical dataset headers", flush=True)

for subject in SUBJECTS:
    data_path = (
        PREPARED_DIR
        / (
            f"{subject}_GLASSdomain_128_"
            "split42_glassseed1.npz"
        )
    )

    try:
        headers = archive_headers(data_path)

        for key, expected_shape in expected_shapes.items():
            actual = headers.get(key, {})

            if actual.get("shape") != expected_shape:
                raise ValueError(
                    f"{key}: expected {expected_shape}, "
                    f"found {actual}"
                )

        dataset_records[subject] = {
            "path": str(data_path),
            "headers": headers,
        }

        print(
            subject,
            "| train: (2520, 8, 128)",
            "| validation: (840, 8, 128)",
            flush=True,
        )

    except (
        OSError,
        ValueError,
        zipfile.BadZipFile,
        EOFError,
    ) as error:
        problems.append(
            f"{subject}: {error}"
        )

        print(
            subject,
            "| NEEDS ATTENTION:",
            error,
            flush=True,
        )


# ------------------------------------------------------------
# 5. Locate existing GLASS and classifier artifacts
# ------------------------------------------------------------

search_scopes = {
    "public_glass": (
        REPLICATION_DIR
        / "pure_glass_8ch_public_validation"
    ),
    "dbnet_no_gan": (
        REPLICATION_DIR
        / "no_gan"
    ),
    "a01_earlier_results": (
        RESULTS_ROOT
        / "A01"
    ),
}

skip_directories = {
    "training_state",
    "generator_checkpoints",
    "gan_trajectories",
    "phase10_control_ablation",
    ".ipynb_checkpoints",
    "__pycache__",
}

allowed_extensions = {
    ".npz",
    ".json",
    ".csv",
    ".h5",
    ".keras",
}

inventory = []
scan_warnings = []

print("\nSaved artifact inventory", flush=True)

for scope_name, search_root in search_scopes.items():
    if not readable_directory(search_root):
        message = (
            f"Folder not accessible: {search_root}"
        )

        scan_warnings.append(message)
        print("\n", scope_name, "|", message)
        continue

    found = []

    for directory, subdirs, filenames in os.walk(
        search_root,
        followlinks=False,
        onerror=lambda error: scan_warnings.append(
            str(error)
        ),
    ):
        subdirs[:] = sorted(
            name
            for name in subdirs
            if name not in skip_directories
        )

        for filename in sorted(filenames):
            path = Path(directory) / filename

            if path.suffix.lower() not in allowed_extensions:
                continue

            found.append(
                {
                    "scope": scope_name,
                    "path": str(path),
                    "relative_path": str(
                        path.relative_to(RESULTS_ROOT)
                    ),
                }
            )

    inventory.extend(found)

    print(
        f"\n{scope_name}: "
        f"{len(found)} candidate files"
    )

    # Prioritize validation and eight-channel filenames
    # when printing examples.
    examples = sorted(
        found,
        key=lambda record: (
            "val" not in record[
                "relative_path"
            ].lower(),
            "8ch" not in record[
                "relative_path"
            ].lower(),
            record["relative_path"],
        ),
    )[:10]

    for record in examples:
        print(" ", record["relative_path"])

    # Inspect one prediction archive's layout per scope.
    # These are headers only, including when a file
    # also contains test arrays.
    sample_archive = next(
        (
            record
            for record in sorted(
                found,
                key=lambda item: item["relative_path"],
            )
            if record["path"].endswith(".npz")
            and "predict" in Path(
                record["path"]
            ).name.lower()
        ),
        None,
    )

    if sample_archive is not None:
        try:
            sample_headers = archive_headers(
                sample_archive["path"]
            )

            sample_archive["headers"] = (
                sample_headers
            )

            print(
                "  Archive layout:",
                Path(sample_archive["path"]).name,
            )

            for key, information in sample_headers.items():
                print(
                    "   ",
                    key,
                    "| shape:",
                    information.get("shape"),
                    "| dtype:",
                    information.get("dtype"),
                )

        except (
            OSError,
            ValueError,
            zipfile.BadZipFile,
            EOFError,
        ) as error:
            scan_warnings.append(
                "Archive inspection failed: "
                f"{sample_archive['path']}: {error}"
            )


# ------------------------------------------------------------
# 6. Locate classifier source without importing TensorFlow
# ------------------------------------------------------------

MODEL_SOURCE = (
    PROJECT_ROOT
    / "code"
    / "EEG-GANet"
    / "github_model.py"
)

model_source_exists = safe_is_file(
    MODEL_SOURCE
)

print("\nDBNet source:", MODEL_SOURCE)
print("Source file exists:", model_source_exists)

if not model_source_exists:
    problems.append(
        f"DBNet source file missing: {MODEL_SOURCE}"
    )


# ------------------------------------------------------------
# 7. Save a new inventory; preserve previous results
# ------------------------------------------------------------

HYBRID_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

unique_suffix = uuid.uuid4().hex[:8]

INVENTORY_PATH = (
    HYBRID_DIR
    / (
        f"step1_inventory_{timestamp}_"
        f"{unique_suffix}.json"
    )
)

report = {
    "created_utc": timestamp,
    "project_root": str(PROJECT_ROOT),
    "purpose": (
        "Inventory for validation-first "
        "DBNet/GLASS experiments"
    ),
    "datasets": dataset_records,
    "candidate_artifacts": inventory,
    "model_source": str(MODEL_SOURCE),
    "problems": problems,
    "scan_warnings": scan_warnings,
    "training_performed": False,
    "prediction_values_loaded": False,
    "artifact_compatibility_verified": False,
}

with INVENTORY_PATH.open(
    "x",
    encoding="utf-8",
) as handle:
    json.dump(
        report,
        handle,
        indent=2,
    )

print("\nSaved inventory:")
print(INVENTORY_PATH)

for warning in scan_warnings:
    print("SCAN NOTE:", warning)

print("\n" + "=" * 60)

if problems:
    print("STEP 1 NEEDS ATTENTION")

    for problem in problems:
        print(problem)

else:
    print("STEP 1 INVENTORY COMPLETED")
    print(
        "Eight canonical dataset headers and "
        "the DBNet source path were found."
    )

print("=" * 60)

print(
    "\nNext: verify saved validation predictions, "

    "split identities and model provenance."
)

print(
    "Candidate files have not yet been certified "
    "as compatible or complete."
)

print(
    "No model training or test evaluation "
    "was performed."
)

In [ ]:
# ============================================================
# STEP 2 — VALIDATION-ONLY DBNET/GLASS COMPATIBILITY AND FUSION
# Run Step 1 first in the same Colab runtime.
# ============================================================

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Verify Step 1 variables
# ------------------------------------------------------------

needed = [
    "PROJECT_ROOT",
    "PREPARED_DIR",
    "RESULTS_ROOT",
    "REPLICATION_DIR",
    "HYBRID_DIR",
    "SUBJECTS",
]

missing = [
    name
    for name in needed
    if name not in globals()
]

if missing:
    raise NameError(
        "Run Step 1 first. Missing: "
        + ", ".join(missing)
    )

PROJECT_ROOT = Path(PROJECT_ROOT)
PREPARED_DIR = Path(PREPARED_DIR)
RESULTS_ROOT = Path(RESULTS_ROOT)
REPLICATION_DIR = Path(REPLICATION_DIR)
HYBRID_DIR = Path(HYBRID_DIR)
SUBJECTS = list(SUBJECTS)

SEEDS = [1, 2, 3]

# Human-readable trial numbers are one-based.
VAL_ONE_BASED = np.array(
    [12, 13, 15, 16, 18, 22, 29],
    dtype=np.int64,
)

# Saved NumPy trial indices are normally zero-based.
VAL_ZERO_BASED = VAL_ONE_BASED - 1

TEMPERATURES = np.array([
    0.25,
    0.35,
    0.50,
    0.75,
    1.00,
    1.50,
    2.00,
    3.00,
    4.00,
])

ALPHAS = np.linspace(
    0.0,
    1.0,
    21,
)


# ------------------------------------------------------------
# 2. Metric and probability functions
# ------------------------------------------------------------

def softmax(values):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    values = (
        values
        - values.max(
            axis=-1,
            keepdims=True,
        )
    )

    exponential = np.exp(values)

    return (
        exponential
        / exponential.sum(
            axis=-1,
            keepdims=True,
        )
    )


def probability_logit(probabilities):
    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=np.float64,
        ),
        1e-6,
        1.0 - 1e-6,
    )

    return (
        np.log(probabilities)
        - np.log1p(-probabilities)
    )


def scale_logits(
    logits,
    temperature,
):
    return softmax(
        logits / float(temperature)
    )


def scale_probabilities(
    probabilities,
    temperature,
):
    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=np.float64,
        ),
        1e-8,
        1.0,
    )

    return softmax(
        np.log(probabilities)
        / float(temperature)
    )


def structured_nll(
    probabilities,
    labels,
):
    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=np.float64,
        ),
        1e-8,
        1.0,
    )

    labels = np.asarray(
        labels,
        dtype=np.float64,
    )

    return float(
        -np.mean(
            np.sum(
                labels
                * np.log(probabilities),
                axis=-1,
            )
        )
    )


def character_correct_by_sequence(
    probabilities,
    labels,
):
    cumulative_probabilities = np.cumsum(
        probabilities,
        axis=1,
    )

    predicted_stimuli = (
        cumulative_probabilities.argmax(
            axis=-1
        )
    )

    true_stimuli = labels.argmax(
        axis=-1
    )

    # A character is correct only when both its
    # row and column are predicted correctly.
    return np.all(
        predicted_stimuli == true_stimuli,
        axis=2,
    )


def calculate_metrics(
    probabilities,
    labels,
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    labels = np.asarray(
        labels,
        dtype=np.int8,
    )

    character_correct = (
        character_correct_by_sequence(
            probabilities,
            labels,
        )
    )

    character_curve = (
        character_correct.mean(axis=0)
    )

    half_sequence_accuracy = float(
        np.mean(
            probabilities.argmax(axis=-1)
            == labels.argmax(axis=-1)
        )
    )

    return {
        "structured_nll": structured_nll(
            probabilities,
            labels,
        ),
        "half_sequence_accuracy":
            half_sequence_accuracy,
        "character_accuracy_4":
            float(character_curve[3]),
        "character_accuracy_7":
            float(character_curve[6]),
        "character_accuracy_10":
            float(character_curve[9]),
        "character_accuracy_curve":
            json.dumps(
                character_curve.tolist()
            ),
    }


# ------------------------------------------------------------
# 3. Locate DBNet validation files
# ------------------------------------------------------------

def find_dbnet_validation_path(
    subject,
    seed,
):
    candidates = [
        (
            REPLICATION_DIR
            / "no_gan"
            / subject
            / "processed_8ch_128_no_gan"
            / f"seed_{seed}"
            / "validation_predictions.npz"
        ),
        (
            REPLICATION_DIR
            / "no_gan"
            / subject
            / "processed_8ch_128_no_gan"
            / f"classifier_seed_{seed}"
            / "validation_predictions.npz"
        ),
    ]

    if subject == "A01":
        candidates.insert(
            0,
            (
                RESULTS_ROOT
                / "A01"
                / "phase11_no_gan_ablation"
                / "processed_8ch_128_no_gan"
                / f"classifier_seed_{seed}"
                / "validation_predictions.npz"
            ),
        )

    found = list(
        dict.fromkeys(
            path
            for path in candidates
            if path.is_file()
        )
    )

    # Search recursively if neither standard layout matches.
    if not found:
        search_root = (
            REPLICATION_DIR
            / "no_gan"
        )

        seed_folder_names = {
            f"seed_{seed}",
            f"classifier_seed_{seed}",
        }

        found = sorted(
            path
            for path in search_root.rglob(
                "validation_predictions.npz"
            )
            if subject in path.parts
            and (
                "processed_8ch_128_no_gan"
                in path.parts
            )
            and any(
                folder in path.parts
                for folder in seed_folder_names
            )
        )

    if len(found) != 1:
        print(
            f"\nCandidate files for "
            f"{subject}, seed {seed}:"
        )

        for path in found:
            print(path)

        raise FileNotFoundError(
            f"{subject}, seed {seed}: "
            f"expected one validation file; "
            f"found {len(found)}."
        )

    return found[0]


def load_dbnet_validation(path):
    """
    Supports both saved formats:

    A01:
        validation_scores
        validation_labels

    A02-A08:
        target_scores or predictions
        true_labels
    """

    with np.load(
        path,
        allow_pickle=False,
    ) as archive:
        keys = list(archive.files)

        label_key = next(
            (
                key
                for key in [
                    "true_labels",
                    "validation_labels",
                    "labels",
                    "y_true",
                ]
                if key in archive.files
            ),
            None,
        )

        score_key = next(
            (
                key
                for key in [
                    "target_scores",
                    "validation_scores",
                    "validation_probabilities",
                    "probabilities",
                    "predictions",
                ]
                if key in archive.files
            ),
            None,
        )

        if label_key is None:
            raise KeyError(
                f"No supported label array in:\n"
                f"{path}\nKeys: {keys}"
            )

        if score_key is None:
            raise KeyError(
                f"No supported score array in:\n"
                f"{path}\nKeys: {keys}"
            )

        saved_labels = np.asarray(
            archive[label_key]
        ).reshape(-1)

        saved_scores = np.asarray(
            archive[score_key]
        )

    if (
        saved_scores.ndim == 2
        and saved_scores.shape[1] == 2
    ):
        target_scores = (
            saved_scores[:, 1]
        )

    elif saved_scores.ndim == 1:
        target_scores = saved_scores

    else:
        raise ValueError(
            f"Unsupported score shape "
            f"{saved_scores.shape} in:\n{path}"
        )

    target_scores = np.asarray(
        target_scores,
        dtype=np.float64,
    ).reshape(-1)

    if target_scores.shape != (840,):
        raise ValueError(
            f"Expected 840 target scores in "
            f"{path}; found {target_scores.shape}"
        )

    if saved_labels.shape != (840,):
        raise ValueError(
            f"Expected 840 labels in "
            f"{path}; found {saved_labels.shape}"
        )

    if not np.all(
        np.isfinite(target_scores)
    ):
        raise ValueError(
            f"Non-finite DBNet scores in {path}"
        )

    if (
        target_scores.min() < 0.0
        or target_scores.max() > 1.0
    ):
        raise ValueError(
            "DBNet scores were expected to be "
            "probabilities, but their range is "
            f"{target_scores.min()} to "
            f"{target_scores.max()} in:\n{path}"
        )

    return (
        target_scores,
        saved_labels.astype(np.int8),
        score_key,
        label_key,
    )


# ------------------------------------------------------------
# 4. Load and align validation predictions
# ------------------------------------------------------------

print("Project:", PROJECT_ROOT)
print("Output:", HYBRID_DIR)
print("Data scope: validation only")

paired_runs = []
glass_subjects = {}
provenance = []

for subject in SUBJECTS:
    print("\n" + "=" * 60)
    print(subject)
    print("=" * 60)

    dataset_path = (
        PREPARED_DIR
        / (
            f"{subject}_GLASSdomain_128_"
            "split42_glassseed1.npz"
        )
    )

    glass_path = (
        REPLICATION_DIR
        / "pure_glass_8ch_public_validation"
        / "frozen_split_fits"
        / (
            f"{subject}_official_"
            "glass_predictions.npz"
        )
    )

    if not dataset_path.is_file():
        raise FileNotFoundError(
            dataset_path
        )

    if not glass_path.is_file():
        raise FileNotFoundError(
            glass_path
        )

    # Load canonical validation labels.
    with np.load(
        dataset_path,
        allow_pickle=False,
    ) as dataset:
        y_validation = np.asarray(
            dataset["y_validation"],
            dtype=np.int8,
        )

        if (
            "validation_trial_indices"
            in dataset.files
        ):
            validation_trials = np.asarray(
                dataset[
                    "validation_trial_indices"
                ],
                dtype=np.int64,
            )

            if np.array_equal(
                validation_trials,
                VAL_ZERO_BASED,
            ):
                trial_index_basis = (
                    "zero_based"
                )

            elif np.array_equal(
                validation_trials,
                VAL_ONE_BASED,
            ):
                trial_index_basis = (
                    "one_based"
                )

            else:
                raise AssertionError(
                    "Unexpected validation trials "
                    f"for {subject}: "
                    f"{validation_trials.tolist()}"
                )

        else:
            trial_index_basis = (
                "not_stored"
            )

    if y_validation.shape != (840,):
        raise ValueError(
            f"{subject}: unexpected "
            f"y_validation shape "
            f"{y_validation.shape}"
        )

    grouped_labels = (
        y_validation.reshape(
            7,
            10,
            2,
            6,
        )
    )

    if not np.all(
        grouped_labels.sum(axis=-1) == 1
    ):
        raise AssertionError(
            f"{subject}: each group of six "
            "must contain exactly one target."
        )

    # Only the validation arrays are accessed.
    with np.load(
        glass_path,
        allow_pickle=False,
    ) as glass_archive:
        glass_probabilities = np.asarray(
            glass_archive[
                "validation_probabilities"
            ],
            dtype=np.float64,
        )

        glass_labels = np.asarray(
            glass_archive[
                "validation_labels"
            ],
            dtype=np.int8,
        )

        beta_shape = tuple(
            np.asarray(
                glass_archive["betaMat"]
            ).shape
        )

    if (
        glass_probabilities.shape
        != (7, 10, 2, 6)
    ):
        raise ValueError(
            f"{subject}: unexpected GLASS "
            f"probability shape "
            f"{glass_probabilities.shape}"
        )

    if beta_shape != (8, 128):
        raise ValueError(
            f"{subject}: unexpected betaMat "
            f"shape {beta_shape}"
        )

    np.testing.assert_array_equal(
        glass_labels,
        grouped_labels,
        err_msg=(
            f"GLASS/canonical label "
            f"mismatch for {subject}"
        ),
    )

    np.testing.assert_allclose(
        glass_probabilities.sum(axis=-1),
        1.0,
        rtol=1e-5,
        atol=1e-5,
        err_msg=(
            f"Invalid GLASS probabilities "
            f"for {subject}"
        ),
    )

    glass_subjects[subject] = {
        "probabilities":
            glass_probabilities,
        "labels":
            grouped_labels,
    }

    for seed in SEEDS:
        prediction_path = (
            find_dbnet_validation_path(
                subject,
                seed,
            )
        )

        (
            target_scores,
            saved_labels,
            score_key,
            label_key,
        ) = load_dbnet_validation(
            prediction_path
        )

        np.testing.assert_array_equal(
            saved_labels,
            y_validation,
            err_msg=(
                f"DBNet/canonical label "
                f"mismatch for {subject}, "
                f"seed {seed}"
            ),
        )

        grouped_target_logits = (
            probability_logit(
                target_scores
            ).reshape(
                7,
                10,
                2,
                6,
            )
        )

        paired_runs.append({
            "subject": subject,
            "classifier_seed": seed,
            "dbnet_logits":
                grouped_target_logits,
            "glass_probabilities":
                glass_probabilities,
            "labels":
                grouped_labels,
        })

        provenance.append({
            "subject": subject,
            "classifier_seed": seed,
            "prediction_path":
                str(prediction_path),
            "score_key": score_key,
            "label_key": label_key,
            "trial_index_basis":
                trial_index_basis,
        })

        print(
            f"seed {seed}"
            f" | score key: {score_key}"
            f" | labels aligned"
            f" | range: "
            f"{target_scores.min():.6f}"
            f" to "
            f"{target_scores.max():.6f}"
        )

if len(paired_runs) != 24:
    raise AssertionError(
        f"Expected 24 paired runs; "
        f"found {len(paired_runs)}"
    )


# ------------------------------------------------------------
# 5. Select global validation temperatures
# ------------------------------------------------------------

all_dbnet_logits = np.concatenate(
    [
        run["dbnet_logits"].reshape(
            -1,
            6,
        )
        for run in paired_runs
    ],
    axis=0,
)

all_dbnet_labels = np.concatenate(
    [
        run["labels"].reshape(
            -1,
            6,
        )
        for run in paired_runs
    ],
    axis=0,
)

all_glass_probabilities = np.concatenate(
    [
        glass_subjects[
            subject
        ]["probabilities"].reshape(
            -1,
            6,
        )
        for subject in SUBJECTS
    ],
    axis=0,
)

all_glass_labels = np.concatenate(
    [
        glass_subjects[
            subject
        ]["labels"].reshape(
            -1,
            6,
        )
        for subject in SUBJECTS
    ],
    axis=0,
)

dbnet_temperature_losses = {
    float(temperature):
        structured_nll(
            scale_logits(
                all_dbnet_logits,
                temperature,
            ),
            all_dbnet_labels,
        )
    for temperature in TEMPERATURES
}

glass_temperature_losses = {
    float(temperature):
        structured_nll(
            scale_probabilities(
                all_glass_probabilities,
                temperature,
            ),
            all_glass_labels,
        )
    for temperature in TEMPERATURES
}

selected_dbnet_temperature = min(
    dbnet_temperature_losses,
    key=dbnet_temperature_losses.get,
)

selected_glass_temperature = min(
    glass_temperature_losses,
    key=glass_temperature_losses.get,
)

for run in paired_runs:
    run["dbnet_probabilities"] = (
        scale_logits(
            run["dbnet_logits"],
            selected_dbnet_temperature,
        )
    )

    run["glass_calibrated"] = (
        scale_probabilities(
            run["glass_probabilities"],
            selected_glass_temperature,
        )
    )


# ------------------------------------------------------------
# 6. Select one global DBNet–GLASS mixture weight
#
# alpha = 0 means DBNet only.
# alpha = 1 means GLASS only.
# ------------------------------------------------------------

alpha_losses = {}

for alpha_value in ALPHAS:
    run_losses = []

    for run in paired_runs:
        fused_probabilities = (
            (1.0 - alpha_value)
            * run["dbnet_probabilities"]
            + alpha_value
            * run["glass_calibrated"]
        )

        run_losses.append(
            structured_nll(
                fused_probabilities,
                run["labels"],
            )
        )

    alpha_losses[
        float(alpha_value)
    ] = float(np.mean(run_losses))

selected_alpha = min(
    alpha_losses,
    key=alpha_losses.get,
)

print(
    "\nValidation-selected global parameters"
)

print(
    "DBNet temperature:",
    selected_dbnet_temperature,
)

print(
    "GLASS temperature:",
    selected_glass_temperature,
)

print(
    "GLASS mixture weight alpha:",
    selected_alpha,
)


# ------------------------------------------------------------
# 7. Calculate validation metrics
# ------------------------------------------------------------

metric_rows = []
complementarity_rows = []

# GLASS has one fitted model per subject.
for subject in SUBJECTS:
    subject_data = (
        glass_subjects[subject]
    )

    calibrated_glass = (
        scale_probabilities(
            subject_data[
                "probabilities"
            ],
            selected_glass_temperature,
        )
    )

    metric_rows.append({
        "subject": subject,
        "classifier_seed": np.nan,
        "model":
            "public_glass_calibrated",
        **calculate_metrics(
            calibrated_glass,
            subject_data["labels"],
        ),
    })

for run in paired_runs:
    fused_probabilities = (
        (1.0 - selected_alpha)
        * run["dbnet_probabilities"]
        + selected_alpha
        * run["glass_calibrated"]
    )

    configurations = [
        (
            "dbnet_binary_posthoc_"
            "six_choice",
            run["dbnet_probabilities"],
        ),
        (
            "validation_score_mixture",
            fused_probabilities,
        ),
    ]

    for (
        configuration_name,
        probabilities,
    ) in configurations:
        metric_rows.append({
            "subject":
                run["subject"],
            "classifier_seed":
                run["classifier_seed"],
            "model":
                configuration_name,
            **calculate_metrics(
                probabilities,
                run["labels"],
            ),
        })

    dbnet_correct = (
        character_correct_by_sequence(
            run["dbnet_probabilities"],
            run["labels"],
        )
    )

    glass_correct = (
        character_correct_by_sequence(
            run["glass_calibrated"],
            run["labels"],
        )
    )

    disagreement = float(
        np.mean(
            run[
                "dbnet_probabilities"
            ].argmax(axis=-1)
            != run[
                "glass_calibrated"
            ].argmax(axis=-1)
        )
    )

    complementarity_rows.append({
        "subject":
            run["subject"],
        "classifier_seed":
            run["classifier_seed"],
        "half_sequence_prediction_disagreement":
            disagreement,
        "oracle_character_accuracy_4":
            float(
                np.mean(
                    dbnet_correct[:, 3]
                    | glass_correct[:, 3]
                )
            ),
        "oracle_character_accuracy_7":
            float(
                np.mean(
                    dbnet_correct[:, 6]
                    | glass_correct[:, 6]
                )
            ),
        "oracle_character_accuracy_10":
            float(
                np.mean(
                    dbnet_correct[:, 9]
                    | glass_correct[:, 9]
                )
            ),
    })


# ------------------------------------------------------------
# 8. Build and save result tables
# ------------------------------------------------------------

metrics_frame = pd.DataFrame(
    metric_rows
)

complementarity_frame = (
    pd.DataFrame(
        complementarity_rows
    )
)

summary_frame = (
    metrics_frame
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        runs=(
            "subject",
            "size",
        ),
        subjects=(
            "subject",
            "nunique",
        ),
        structured_nll=(
            "structured_nll",
            "mean",
        ),
        half_sequence_accuracy=(
            "half_sequence_accuracy",
            "mean",
        ),
        accuracy_4_sequences=(
            "character_accuracy_4",
            "mean",
        ),
        accuracy_7_sequences=(
            "character_accuracy_7",
            "mean",
        ),
        accuracy_10_sequences=(
            "character_accuracy_10",
            "mean",
        ),
    )
)

HYBRID_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

STAGE2_METRICS_PATH = (
    HYBRID_DIR
    / (
        f"step2_validation_metrics_"
        f"{timestamp}.csv"
    )
)

STAGE2_SUMMARY_PATH = (
    HYBRID_DIR
    / (
        f"step2_validation_summary_"
        f"{timestamp}.csv"
    )
)

STAGE2_COMPLEMENTARITY_PATH = (
    HYBRID_DIR
    / (
        f"step2_complementarity_"
        f"{timestamp}.csv"
    )
)

STAGE2_SELECTION_PATH = (
    HYBRID_DIR
    / (
        f"step2_selection_"
        f"{timestamp}.json"
    )
)

metrics_frame.to_csv(
    STAGE2_METRICS_PATH,
    index=False,
)

summary_frame.to_csv(
    STAGE2_SUMMARY_PATH,
    index=False,
)

complementarity_frame.to_csv(
    STAGE2_COMPLEMENTARITY_PATH,
    index=False,
)

selection_report = {
    "data_scope":
        "validation only",
    "subjects":
        SUBJECTS,
    "classifier_seeds":
        SEEDS,
    "selected_dbnet_temperature":
        float(
            selected_dbnet_temperature
        ),
    "selected_glass_temperature":
        float(
            selected_glass_temperature
        ),
    "selected_glass_mixture_weight_alpha":
        float(selected_alpha),
    "temperature_grid":
        TEMPERATURES.tolist(),
    "alpha_grid":
        ALPHAS.tolist(),
    "dbnet_temperature_nll":
        dbnet_temperature_losses,
    "glass_temperature_nll":
        glass_temperature_losses,
    "alpha_nll":
        alpha_losses,
    "provenance":
        provenance,
    "test_arrays_accessed":
        False,
    "model_training_performed":
        False,
    "interpretation": (
        "The mixture is a validation "
        "diagnostic. DBNet uses post-hoc "
        "six-choice normalization of the "
        "saved binary classifier scores."
    ),
}

with STAGE2_SELECTION_PATH.open(
    "x",
    encoding="utf-8",
) as handle:
    json.dump(
        selection_report,
        handle,
        indent=2,
    )


# ------------------------------------------------------------
# 9. Display final validation results
# ------------------------------------------------------------

print("\nValidation summary")

print(
    summary_frame
    .round(4)
    .to_string(index=False)
)

print(
    "\nComplementarity summary"
)

complementarity_columns = [
    "half_sequence_prediction_disagreement",
    "oracle_character_accuracy_4",
    "oracle_character_accuracy_7",
    "oracle_character_accuracy_10",
]

print(
    complementarity_frame[
        complementarity_columns
    ]
    .mean()
    .round(4)
    .to_string()
)

print("\nSaved:")
print(STAGE2_METRICS_PATH)
print(STAGE2_SUMMARY_PATH)
print(STAGE2_COMPLEMENTARITY_PATH)
print(STAGE2_SELECTION_PATH)

print("\n" + "=" * 60)
print("STEP 2 COMPLETED")
print(
    "All eight subjects and 24 DBNet "
    "validation runs were aligned."
)
print(
    "Only validation predictions were "
    "evaluated."
)
print(
    "The mixture is a diagnostic result, "
    "not the trained hybrid model."
)
print("=" * 60)

In [ ]:
# ============================================================
# STEP 3A — STRUCTURED SIX-CHOICE DBNET SETUP CHECK
#
# Run Step 1 first in the same Colab runtime.
# No training or test evaluation is performed.
# ============================================================

import importlib.util
import json
import sys

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorflow as tf


# ------------------------------------------------------------
# 1. Check required variables
# ------------------------------------------------------------

required_variables = [
    "PROJECT_ROOT",
    "PREPARED_DIR",
    "HYBRID_DIR",
    "SUBJECTS",
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Run Step 1 first. Missing: "
        + ", ".join(missing_variables)
    )

PROJECT_ROOT = Path(PROJECT_ROOT)
PREPARED_DIR = Path(PREPARED_DIR)
HYBRID_DIR = Path(HYBRID_DIR)
SUBJECTS = list(SUBJECTS)


# ------------------------------------------------------------
# 2. Frozen structured-training protocol
# ------------------------------------------------------------

NUMBER_OF_CHOICES = 6
NUMBER_OF_CHANNELS = 8
NUMBER_OF_SAMPLES = 128

LEARNING_RATE = 0.0009
GROUP_BATCH_SIZE = 32
MAXIMUM_EPOCHS = 1000
PATIENCE = 50


# ------------------------------------------------------------
# 3. Load the authors' DBNet-V2 implementation
# ------------------------------------------------------------

CODE_DIR = (
    PROJECT_ROOT
    / "code"
    / "EEG-GANet"
)

MODEL_SOURCE = (
    CODE_DIR
    / "github_model.py"
)

UTILITY_SOURCE = (
    CODE_DIR
    / "github_Utils.py"
)

if not MODEL_SOURCE.is_file():
    raise FileNotFoundError(
        MODEL_SOURCE
    )

if not UTILITY_SOURCE.is_file():
    raise FileNotFoundError(
        UTILITY_SOURCE
    )

if str(CODE_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(CODE_DIR),
    )

module_spec = (
    importlib.util.spec_from_file_location(
        "github_model_dbnet_glass_step3a",
        MODEL_SOURCE,
    )
)

github_model = (
    importlib.util.module_from_spec(
        module_spec
    )
)

module_spec.loader.exec_module(
    github_model
)

EEG_DBNet_V2 = (
    github_model.EEG_DBNet_V2
)


# ------------------------------------------------------------
# 4. Load each six-stimulus group
# ------------------------------------------------------------

def load_grouped_split(
    subject,
    split,
):
    data_path = (
        PREPARED_DIR
        / (
            f"{subject}_GLASSdomain_128_"
            "split42_glassseed1.npz"
        )
    )

    if not data_path.is_file():
        raise FileNotFoundError(
            data_path
        )

    with np.load(
        data_path,
        allow_pickle=False,
    ) as archive:
        X = np.asarray(
            archive[
                f"X_{split}_8ch"
            ],
            dtype=np.float32,
        )

        y = np.asarray(
            archive[f"y_{split}"],
            dtype=np.int8,
        )

    if split == "train":
        expected_epochs = 2520
    else:
        expected_epochs = 840

    if X.shape != (
        expected_epochs,
        8,
        128,
    ):
        raise ValueError(
            f"{subject}/{split}: "
            f"unexpected X shape {X.shape}"
        )

    if y.shape != (
        expected_epochs,
    ):
        raise ValueError(
            f"{subject}/{split}: "
            f"unexpected y shape {y.shape}"
        )

    # Each consecutive six epochs represent
    # the six row or six column candidates.
    grouped_X = X.reshape(
        -1,
        6,
        8,
        128,
        1,
    )

    grouped_y = y.reshape(
        -1,
        6,
    )

    if not np.all(
        grouped_y.sum(axis=1) == 1
    ):
        raise AssertionError(
            f"{subject}/{split}: every "
            "six-stimulus group must contain "
            "exactly one target."
        )

    return (
        grouped_X,
        grouped_y.astype(np.float32),
    )


# ------------------------------------------------------------
# 5. Create a shared single-stimulus DBNet scorer
# ------------------------------------------------------------

def build_epoch_scorer():
    public_binary_model = (
        EEG_DBNet_V2(
            NumFilter=8,
            SamplingFrequency=128,
            NumChannels=8,
            FilterScaler=2,
            NumClasses=2,
            DropoutRate=0.5,
        ).build_model()
    )

    # Public final layers should be:
    # Concatenate -> Dense -> sigmoid
    if not isinstance(
        public_binary_model.layers[-3],
        tf.keras.layers.Concatenate,
    ):
        raise AssertionError(
            "Unexpected public DBNet-V2 "
            "architecture. Expected the third "
            "layer from the end to be Concatenate."
        )

    feature_extractor = (
        tf.keras.Model(
            inputs=
                public_binary_model.input,
            outputs=
                public_binary_model
                .layers[-3]
                .output,
            name=
                "dbnet_v2_feature_extractor",
        )
    )

    epoch_input = (
        tf.keras.Input(
            shape=(8, 128, 1),
            name="single_stimulus_epoch",
        )
    )

    features = feature_extractor(
        epoch_input
    )

    # One unnormalized target score for
    # every stimulus epoch.
    stimulus_score = (
        tf.keras.layers.Dense(
            1,
            kernel_constraint=
                tf.keras.constraints
                .max_norm(0.25),
            name="stimulus_score",
        )(features)
    )

    return tf.keras.Model(
        inputs=epoch_input,
        outputs=stimulus_score,
        name="dbnet_v2_epoch_scorer",
    )


# ------------------------------------------------------------
# 6. Apply the same scorer to all six candidates
# ------------------------------------------------------------

def build_structured_dbnet():
    epoch_scorer = (
        build_epoch_scorer()
    )

    group_input = (
        tf.keras.Input(
            shape=(
                6,
                8,
                128,
                1,
            ),
            name="six_stimulus_group",
        )
    )

    stimulus_scores = (
        tf.keras.layers.TimeDistributed(
            epoch_scorer,
            name="shared_epoch_scorer",
        )(group_input)
    )

    six_choice_logits = (
        tf.keras.layers.Reshape(
            (6,),
            name="six_choice_logits",
        )(stimulus_scores)
    )

    six_choice_probabilities = (
        tf.keras.layers.Softmax(
            axis=-1,
            name=
                "six_choice_probabilities",
        )(six_choice_logits)
    )

    model = tf.keras.Model(
        inputs=group_input,
        outputs=six_choice_probabilities,
        name="structured_dbnet_v2",
    )

    model.compile(
        optimizer=
            tf.keras.optimizers.Adam(
                learning_rate=
                    LEARNING_RATE
            ),
        loss=
            tf.keras.losses
            .CategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics
            .CategoricalAccuracy(
                name=
                    "six_choice_accuracy"
            )
        ],
    )

    return model


# ------------------------------------------------------------
# 7. Verify all subject data
# ------------------------------------------------------------

print("Project:", PROJECT_ROOT)
print("TensorFlow:", tf.__version__)

print(
    "GPU:",
    tf.config.list_physical_devices(
        "GPU"
    ),
)

print(
    "DBNet source:",
    MODEL_SOURCE,
)

dataset_summary = {}

for subject in SUBJECTS:
    (
        X_training,
        y_training,
    ) = load_grouped_split(
        subject,
        "train",
    )

    (
        X_validation,
        y_validation,
    ) = load_grouped_split(
        subject,
        "validation",
    )

    dataset_summary[subject] = {
        "training_shape":
            list(X_training.shape),
        "validation_shape":
            list(X_validation.shape),
        "training_groups":
            int(len(X_training)),
        "validation_groups":
            int(len(X_validation)),
    }

    print(
        subject,
        "| train:",
        X_training.shape,
        "| validation:",
        X_validation.shape,
    )


# ------------------------------------------------------------
# 8. Build and test the model
# ------------------------------------------------------------

tf.keras.backend.clear_session()

tf.keras.utils.set_random_seed(
    1
)

structured_model = (
    build_structured_dbnet()
)

(
    X_example,
    y_example,
) = load_grouped_split(
    "A01",
    "train",
)

example_probabilities = (
    structured_model(
        X_example[:2],
        training=False,
    )
    .numpy()
)

if example_probabilities.shape != (
    2,
    6,
):
    raise AssertionError(
        "Unexpected model output shape: "
        f"{example_probabilities.shape}"
    )

np.testing.assert_allclose(
    example_probabilities.sum(
        axis=1
    ),
    1.0,
    rtol=1e-5,
    atol=1e-5,
)

if not np.all(
    np.isfinite(
        example_probabilities
    )
):
    raise AssertionError(
        "Forward pass produced "
        "non-finite probabilities."
    )


# ------------------------------------------------------------
# 9. Compare parameter counts
# ------------------------------------------------------------

original_binary_model = (
    EEG_DBNet_V2(
        NumFilter=8,
        SamplingFrequency=128,
        NumChannels=8,
        FilterScaler=2,
        NumClasses=2,
        DropoutRate=0.5,
    ).build_model()
)

original_binary_parameters = (
    original_binary_model.count_params()
)

structured_parameters = (
    structured_model.count_params()
)


# ------------------------------------------------------------
# 10. Save setup manifest
# ------------------------------------------------------------

HYBRID_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

SETUP_MANIFEST_PATH = (
    HYBRID_DIR
    / f"step3a_setup_{timestamp}.json"
)

setup_manifest = {
    "created_utc":
        timestamp,
    "model":
        "structured six-choice EEG-DBNet-V2",
    "public_model_source":
        str(MODEL_SOURCE),
    "structured_model_parameters":
        structured_parameters,
    "original_binary_model_parameters":
        original_binary_parameters,
    "input_shape":
        list(
            structured_model.input_shape
        ),
    "output_shape":
        list(
            structured_model.output_shape
        ),
    "number_of_choices":
        NUMBER_OF_CHOICES,
    "learning_rate":
        LEARNING_RATE,
    "group_batch_size":
        GROUP_BATCH_SIZE,
    "maximum_epochs":
        MAXIMUM_EPOCHS,
    "patience":
        PATIENCE,
    "datasets":
        dataset_summary,
    "forward_pass_verified":
        True,
    "training_performed":
        False,
    "test_data_accessed":
        False,
}

with SETUP_MANIFEST_PATH.open(
    "x",
    encoding="utf-8",
) as handle:
    json.dump(
        setup_manifest,
        handle,
        indent=2,
    )


# ------------------------------------------------------------
# 11. Final output
# ------------------------------------------------------------

print(
    "\nStructured DBNet input:",
    structured_model.input_shape,
)

print(
    "Structured DBNet output:",
    structured_model.output_shape,
)

print(
    "Structured DBNet parameters:",
    structured_parameters,
)

print(
    "Original binary DBNet parameters:",
    original_binary_parameters,
)

print(
    "\nExample probabilities:"
)

print(
    np.round(
        example_probabilities,
        6,
    )
)

print(
    "Probability row sums:",
    example_probabilities.sum(
        axis=1
    ),
)

print(
    "\nSaved:",
    SETUP_MANIFEST_PATH,
)

print(
    "\nSTEP 3A PASSED"
)

print(
    "The shared DBNet scorer and "
    "six-choice softmax are ready."
)

print(
    "No training or test evaluation "
    "was performed."
)

In [ ]:
# ============================================================
# STEP 3B — TRAIN STRUCTURED DBNET, SEED 1, A01-A08
# Requires Step 3A in the same runtime.
# Uses training and validation data only. Test data are untouched.
# Completed subject runs are verified and skipped after reconnects.
# ============================================================

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf


required_names = [
    "PROJECT_ROOT", "PREPARED_DIR", "HYBRID_DIR", "SUBJECTS",
    "build_structured_dbnet", "load_grouped_split",
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Run Step 1 and Step 3A in this runtime first. Missing: "
        + ", ".join(missing_names)
    )

PROJECT_ROOT = Path(PROJECT_ROOT)
PREPARED_DIR = Path(PREPARED_DIR)
HYBRID_DIR = Path(HYBRID_DIR)
SUBJECTS = list(SUBJECTS)

TRAINING_SEEDS = [1]
MAXIMUM_EPOCHS = 1000
PATIENCE = 50
MINIMUM_DELTA = 1e-4
GROUP_BATCH_SIZE = 32
TRAINING_ROOT = HYBRID_DIR / "structured_dbnet_training"
TRAINING_ROOT.mkdir(parents=True, exist_ok=True)

# Validation-only reference produced in Step 2 from the same subjects/split.
# For NLL, a negative delta is better. For accuracies, a positive delta is better.
BINARY_POSTHOC_REFERENCE = {
    "structured_nll": 0.9397,
    "half_sequence_accuracy": 0.6688,
    "character_accuracy_4": 0.8631,
    "character_accuracy_7": 0.9464,
    "character_accuracy_10": 0.9821,
}


def write_json_safely(data, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".temporary")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2)
    temporary.replace(path)


def structured_validation_metrics(probabilities, labels):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    if probabilities.shape != (140, 6) or labels.shape != (140, 6):
        raise ValueError(
            f"Expected (140, 6) probabilities and labels; found "
            f"{probabilities.shape}, {labels.shape}"
        )
    np.testing.assert_allclose(
        probabilities.sum(axis=1), 1.0, rtol=1e-5, atol=1e-5
    )
    grouped_p = probabilities.reshape(7, 10, 2, 6)
    grouped_y = labels.reshape(7, 10, 2, 6)
    cumulative = np.cumsum(grouped_p, axis=1)
    predicted = cumulative.argmax(axis=-1)
    truth = grouped_y.argmax(axis=-1)
    character_curve = np.all(predicted == truth, axis=2).mean(axis=0)
    clipped = np.clip(probabilities, 1e-8, 1.0)
    nll_value = float(-np.mean(np.sum(labels * np.log(clipped), axis=1)))
    return {
        "structured_nll": nll_value,
        "half_sequence_accuracy": float(
            np.mean(probabilities.argmax(axis=1) == labels.argmax(axis=1))
        ),
        "character_accuracy_4": float(character_curve[3]),
        "character_accuracy_7": float(character_curve[6]),
        "character_accuracy_10": float(character_curve[9]),
        "character_accuracy_curve": character_curve.tolist(),
    }


def verified_completed_run(run_dir):
    required = [
        run_dir / "completed.json",
        run_dir / "validation_metrics.json",
        run_dir / "validation_predictions.npz",
        run_dir / "best.weights.h5",
        run_dir / "history.csv",
    ]
    if not all(path.is_file() for path in required):
        return None
    try:
        with (run_dir / "validation_metrics.json").open(
            "r", encoding="utf-8"
        ) as handle:
            metrics = json.load(handle)
        with np.load(
            run_dir / "validation_predictions.npz", allow_pickle=False
        ) as archive:
            p = np.asarray(archive["probabilities"])
            y = np.asarray(archive["labels"])
        if p.shape != (140, 6) or y.shape != (140, 6):
            return None
        np.testing.assert_allclose(p.sum(axis=1), 1.0, rtol=1e-5, atol=1e-5)
        return metrics
    except (
        OSError,
        ValueError,
        KeyError,
        AssertionError,
        json.JSONDecodeError,
    ):
        return None


class CompactProgress(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        epoch_number = epoch + 1
        if epoch_number == 1 or epoch_number % 25 == 0:
            print(
                f"epoch {epoch_number:4d} | "
                f"loss {logs.get('loss', float('nan')):.4f} | "
                f"val_loss {logs.get('val_loss', float('nan')):.4f} | "
                f"val_accuracy "
                f"{logs.get('val_six_choice_accuracy', float('nan')):.4f}",
                flush=True,
            )


print("Project:", PROJECT_ROOT)
print("Training output:", TRAINING_ROOT)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
print("Subjects:", SUBJECTS)
print("Training seeds:", TRAINING_SEEDS)
print("Maximum epochs:", MAXIMUM_EPOCHS)
print("Patience:", PATIENCE)

all_results = []

for subject in SUBJECTS:
    for seed in TRAINING_SEEDS:
        print("\n" + "=" * 68)
        print(f"{subject} | structured DBNet | seed {seed}")
        print("=" * 68)

        run_dir = TRAINING_ROOT / subject / f"seed_{seed}"
        run_dir.mkdir(parents=True, exist_ok=True)
        completed_metrics = verified_completed_run(run_dir)
        if completed_metrics is not None:
            print("Skipping verified completed run.")
            all_results.append(completed_metrics)
            continue

        X_train, y_train = load_grouped_split(subject, "train")
        X_validation, y_validation = load_grouped_split(subject, "validation")
        if X_train.shape != (420, 6, 8, 128, 1):
            raise ValueError(f"Unexpected training shape: {X_train.shape}")
        if X_validation.shape != (140, 6, 8, 128, 1):
            raise ValueError(f"Unexpected validation shape: {X_validation.shape}")

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)
        model = build_structured_dbnet()
        best_weights_path = run_dir / "best.weights.h5"
        history_path = run_dir / "history.csv"

        callbacks = [
            tf.keras.callbacks.TerminateOnNaN(),
            tf.keras.callbacks.ModelCheckpoint(
                filepath=str(best_weights_path),
                monitor="val_loss",
                mode="min",
                save_best_only=True,
                save_weights_only=True,
                verbose=0,
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=PATIENCE,
                min_delta=MINIMUM_DELTA,
                restore_best_weights=True,
                verbose=1,
            ),
            tf.keras.callbacks.CSVLogger(
                str(history_path), append=False
            ),
            CompactProgress(),
        ]

        start_time = time.perf_counter()
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_validation, y_validation),
            epochs=MAXIMUM_EPOCHS,
            batch_size=GROUP_BATCH_SIZE,
            shuffle=True,
            callbacks=callbacks,
            verbose=0,
        )
        training_seconds = float(time.perf_counter() - start_time)

        if not best_weights_path.is_file():
            raise FileNotFoundError(
                f"Best weights were not saved for {subject}, seed {seed}"
            )
        model.load_weights(best_weights_path)
        validation_probabilities = model.predict(
            X_validation, batch_size=GROUP_BATCH_SIZE, verbose=0
        ).astype(np.float32)
        calculated = structured_validation_metrics(
            validation_probabilities, y_validation
        )

        validation_losses = np.asarray(history.history["val_loss"])
        best_epoch = int(validation_losses.argmin() + 1)
        epochs_trained = int(len(validation_losses))
        result = {
            "subject": subject,
            "classifier_seed": seed,
            "configuration": "structured_dbnet_v2_six_choice",
            "parameters": int(model.count_params()),
            "best_epoch": best_epoch,
            "epochs_trained": epochs_trained,
            "training_seconds": training_seconds,
            "training_minutes": training_seconds / 60.0,
            **calculated,
            "test_data_accessed": False,
        }

        np.savez_compressed(
            run_dir / "validation_predictions.npz",
            probabilities=validation_probabilities,
            labels=y_validation.astype(np.int8),
            best_epoch=np.int32(best_epoch),
        )
        write_json_safely(result, run_dir / "validation_metrics.json")
        write_json_safely(
            {
                "status": "completed",
                "subject": subject,
                "classifier_seed": seed,
                "completed_utc": datetime.now(timezone.utc).isoformat(),
                "best_weights": str(best_weights_path),
                "validation_predictions": str(
                    run_dir / "validation_predictions.npz"
                ),
                "test_data_accessed": False,
            },
            run_dir / "completed.json",
        )
        all_results.append(result)
        print(
            f"Completed | best epoch: {best_epoch} | "
            f"minutes: {training_seconds / 60.0:.2f} | "
            f"4-seq: {calculated['character_accuracy_4']:.4f} | "
            f"7-seq: {calculated['character_accuracy_7']:.4f} | "
            f"10-seq: {calculated['character_accuracy_10']:.4f}"
        )

results_frame = pd.DataFrame(all_results).sort_values(
    ["subject", "classifier_seed"]
)
summary = {
    "configuration": "structured_dbnet_v2_six_choice",
    "subjects": SUBJECTS,
    "classifier_seeds": TRAINING_SEEDS,
    "completed_runs": int(len(results_frame)),
    "mean_structured_nll": float(results_frame["structured_nll"].mean()),
    "mean_half_sequence_accuracy": float(
        results_frame["half_sequence_accuracy"].mean()
    ),
    "mean_character_accuracy_4": float(
        results_frame["character_accuracy_4"].mean()
    ),
    "mean_character_accuracy_7": float(
        results_frame["character_accuracy_7"].mean()
    ),
    "mean_character_accuracy_10": float(
        results_frame["character_accuracy_10"].mean()
    ),
    "total_training_minutes": float(results_frame["training_minutes"].sum()),
    "binary_posthoc_validation_reference": BINARY_POSTHOC_REFERENCE,
    "test_data_accessed": False,
}

summary["delta_vs_binary_posthoc"] = {
    "structured_nll": (
        summary["mean_structured_nll"]
        - BINARY_POSTHOC_REFERENCE["structured_nll"]
    ),
    "half_sequence_accuracy": (
        summary["mean_half_sequence_accuracy"]
        - BINARY_POSTHOC_REFERENCE["half_sequence_accuracy"]
    ),
    "character_accuracy_4": (
        summary["mean_character_accuracy_4"]
        - BINARY_POSTHOC_REFERENCE["character_accuracy_4"]
    ),
    "character_accuracy_7": (
        summary["mean_character_accuracy_7"]
        - BINARY_POSTHOC_REFERENCE["character_accuracy_7"]
    ),
    "character_accuracy_10": (
        summary["mean_character_accuracy_10"]
        - BINARY_POSTHOC_REFERENCE["character_accuracy_10"]
    ),
}

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RESULTS_CSV_PATH = TRAINING_ROOT / f"seed1_validation_results_{timestamp}.csv"
SUMMARY_JSON_PATH = TRAINING_ROOT / f"seed1_validation_summary_{timestamp}.json"
results_frame.to_csv(RESULTS_CSV_PATH, index=False)
write_json_safely(summary, SUMMARY_JSON_PATH)

print("\nStructured DBNet seed-1 validation results")
print(
    results_frame[
        [
            "subject", "best_epoch", "training_minutes",
            "structured_nll", "half_sequence_accuracy",
            "character_accuracy_4", "character_accuracy_7",
            "character_accuracy_10",
        ]
    ].round(4).to_string(index=False)
)
print("\nMean results")
for key, value in summary.items():
    if key.startswith("mean_") or key == "total_training_minutes":
        print(key, ":", round(value, 4))
print("\nDelta versus Step-2 binary DBNet post-hoc baseline")
print("Negative is better only for NLL; positive is better for accuracies.")
for key, value in summary["delta_vs_binary_posthoc"].items():
    print(key, ":", round(value, 4))
print("\nSaved:")
print(RESULTS_CSV_PATH)
print(SUMMARY_JSON_PATH)
print("\nSTEP 3B COMPLETED")
print("One structured DBNet seed was trained for all eight subjects.")
print("Test data were not loaded or evaluated.")


In [ ]:
"""Colab Step 3A: train DBNet + 28 RTGP interaction features, seed 1.

This is an exploratory validation-only screen over A01-A08. It is standalone
after Drive is mounted and Steps 1-2 have completed. Completed subject runs are
verified and skipped, so the cell can be rerun safely after a disconnection.

The public DBNet source and the source-derived interaction feature files are
read but never changed. Test EEG, labels, predictions, and metrics are not read.
"""

import hashlib
import importlib.util
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf


# Leave None unless automatic project discovery fails.
PROJECT_ROOT_OVERRIDE = None

SHARED_ID = "1un2SaLv7_DvXerUlPtzKSNZ_j-mu1qFQ"
PROJECT_NAME = "EEG_GANet_Reproduction"
SUBJECTS = [f"A{i:02d}" for i in range(1, 9)]
TRAINING_SEED = 1

DBNET_SHA256 = "f3e570d12de34080ea13003a37bf9a2493b93f1a2376631e8224a40942231cd5"
RTGP_ARCHIVE_SHA256 = "d4b72ceb330e4db7f5c8761f8b487e097bc193d54972a4cabee6d8825bd1c037"
RTGP_COMMIT = "7c6e274b00bf9ecde708af3178131bc065245c74"

NUMBER_OF_CHOICES = 6
NUMBER_OF_CHANNELS = 8
NUMBER_OF_SAMPLES = 128
NUMBER_OF_INTERACTIONS = 28
LEARNING_RATE = 0.0009
MAXIMUM_EPOCHS = 1000
PATIENCE = 50
MINIMUM_DELTA = 1e-4
GROUP_BATCH_SIZE = 32

MODEL_PROTOCOL = {
    "model": "structured_dbnet_plus_rtgp_interactions",
    "dbnet_sha256": DBNET_SHA256,
    "rtgp_commit": RTGP_COMMIT,
    "rtgp_archive_sha256": RTGP_ARCHIVE_SHA256,
    "subjects": SUBJECTS,
    "classifier_seed": TRAINING_SEED,
    "choices": NUMBER_OF_CHOICES,
    "channels": NUMBER_OF_CHANNELS,
    "samples": NUMBER_OF_SAMPLES,
    "interaction_features": NUMBER_OF_INTERACTIONS,
    "interaction_branch": "shared linear, no bias, zero initialization",
    "interaction_logit_scaling": 1.0 / NUMBER_OF_INTERACTIONS,
    "additional_feature_standardization": False,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "loss": "categorical_crossentropy",
    "maximum_epochs": MAXIMUM_EPOCHS,
    "patience": PATIENCE,
    "minimum_delta": MINIMUM_DELTA,
    "batch_size_groups": GROUP_BATCH_SIZE,
    "monitor": "val_loss",
    "restore_best_weights": True,
    "test_access": False,
}
MODEL_PROTOCOL_ID = hashlib.sha256(
    json.dumps(MODEL_PROTOCOL, sort_keys=True, separators=(",", ":")).encode()
).hexdigest()


def require(condition, message):
    if not condition:
        raise AssertionError(message)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json_safely(data, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".temporary")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2, allow_nan=False)
    temporary.replace(path)


def save_npz_safely(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.stem + ".temporary.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)


def project_valid(path):
    path = Path(path)
    return (
        path.joinpath(
            "GLASS_GANet/prepared_data/"
            "A01_GLASSdomain_128_split42_glassseed1.npz"
        ).is_file()
        and path.joinpath("code/EEG-GANet/github_model.py").is_file()
    )


def resolve_project():
    if PROJECT_ROOT_OVERRIDE:
        candidate = Path(PROJECT_ROOT_OVERRIDE)
        require(project_valid(candidate), f"Invalid PROJECT_ROOT_OVERRIDE: {candidate}")
        return candidate

    previous = globals().get("PROJECT_ROOT")
    if previous and project_valid(Path(previous)):
        return Path(previous)

    content = Path("/content")
    mounts = [content / "drive"] + sorted(content.glob("eeg_drive*"))
    candidates = []
    for mount in mounts:
        candidates.extend([
            mount / ".shortcut-targets-by-id" / SHARED_ID / PROJECT_NAME,
            mount / "MyDrive" / PROJECT_NAME,
        ])
    for candidate in candidates:
        if project_valid(candidate):
            return candidate
    raise FileNotFoundError(
        "Project not visible. Mount the correct Drive account, or set "
        "PROJECT_ROOT_OVERRIDE.\nChecked:\n" +
        "\n".join(str(item) for item in candidates)
    )


def latest_step2_protocol(output_root):
    candidates = sorted(output_root.glob("step2_architecture_protocol_*.json"))
    require(candidates, "Step-2 architecture protocol not found")
    for path in reversed(candidates):
        with path.open("r", encoding="utf-8") as handle:
            record = json.load(handle)
        if record.get("step") == "interaction_step2" and record.get("status") == "passed":
            audit = record.get("architecture_audit", {})
            require(audit.get("control_parameters") == 3961,
                    "Step-2 control parameter count differs")
            require(audit.get("interaction_model_parameters") == 3989,
                    "Step-2 interaction-model parameter count differs")
            require(audit.get("added_trainable_parameters") == 28,
                    "Step-2 added parameter count differs")
            require(audit.get("dbnet_backbone_initial_weight_max_difference") == 0.0,
                    "Step-2 did not certify identical initial DBNet weights")
            require(audit.get("initial_probability_max_difference") == 0.0,
                    "Step-2 did not certify identical initial predictions")
            require(record.get("training_performed") is False,
                    "Step-2 unexpectedly reports fitting")
            require(record.get("test_EEG_labels_predictions_read") is False,
                    "Step-2 unexpectedly reports test access")
            return path
    raise AssertionError("No passed Step-2 protocol with the expected audit was found")


def load_public_dbnet(project):
    model_dir = project / "code/EEG-GANet"
    source = model_dir / "github_model.py"
    require(file_sha256(source) == DBNET_SHA256,
            "DBNet source hash differs from the certified source")
    if str(model_dir) not in sys.path:
        sys.path.insert(0, str(model_dir))
    spec = importlib.util.spec_from_file_location(
        "github_model_interaction_step3a", source
    )
    require(spec is not None and spec.loader is not None,
            f"Cannot import {source}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    require(hasattr(module, "EEG_DBNet_V2"), "EEG_DBNet_V2 is missing")
    return module.EEG_DBNet_V2, source


def load_grouped_subject(prepared_dir, feature_dir, subject):
    data_path = prepared_dir / f"{subject}_GLASSdomain_128_split42_glassseed1.npz"
    feature_path = feature_dir / f"{subject}_rtgp_correlations_128_split42.npz"
    require(data_path.is_file(), f"Missing canonical dataset: {data_path}")
    require(feature_path.is_file(), f"Missing feature file: {feature_path}")

    # Deliberately request train/validation keys only.
    with np.load(data_path, allow_pickle=False) as data:
        x_train = np.asarray(data["X_train_8ch"], dtype=np.float32)
        y_train = np.asarray(data["y_train"], dtype=np.int8).reshape(-1)
        x_validation = np.asarray(data["X_validation_8ch"], dtype=np.float32)
        y_validation = np.asarray(data["y_validation"], dtype=np.int8).reshape(-1)
        train_trials = np.asarray(data["train_trial_indices"]).reshape(-1)
        validation_trials = np.asarray(data["validation_trial_indices"]).reshape(-1)

    with np.load(feature_path, allow_pickle=False) as features:
        x0_train = np.asarray(features["X0_train"], dtype=np.float32)
        x0_validation = np.asarray(features["X0_validation"], dtype=np.float32)
        feature_y_train = np.asarray(features["y_train"], dtype=np.int8).reshape(-1)
        feature_y_validation = np.asarray(
            features["y_validation"], dtype=np.int8
        ).reshape(-1)
        feature_train_trials = np.asarray(features["train_trial_indices"]).reshape(-1)
        feature_validation_trials = np.asarray(
            features["validation_trial_indices"]
        ).reshape(-1)
        source_commit = str(np.asarray(features["source_commit"]).item())
        source_hash = str(np.asarray(features["source_archive_sha256"]).item())
        pair_names = np.asarray(features["channel_pair_names"]).astype(str)

    require(x_train.shape == (2520, 8, 128),
            f"{subject}: unexpected train EEG shape {x_train.shape}")
    require(x_validation.shape == (840, 8, 128),
            f"{subject}: unexpected validation EEG shape {x_validation.shape}")
    require(x0_train.shape == (2520, 28),
            f"{subject}: unexpected train interaction shape {x0_train.shape}")
    require(x0_validation.shape == (840, 28),
            f"{subject}: unexpected validation interaction shape {x0_validation.shape}")
    require(np.isfinite(x_train).all() and np.isfinite(x_validation).all(),
            f"{subject}: non-finite EEG")
    require(np.isfinite(x0_train).all() and np.isfinite(x0_validation).all(),
            f"{subject}: non-finite interaction feature")
    np.testing.assert_array_equal(y_train, feature_y_train)
    np.testing.assert_array_equal(y_validation, feature_y_validation)
    np.testing.assert_array_equal(train_trials, feature_train_trials)
    np.testing.assert_array_equal(validation_trials, feature_validation_trials)
    require(np.intersect1d(train_trials, validation_trials).size == 0,
            f"{subject}: train/validation trial overlap")
    require(source_commit == RTGP_COMMIT, f"{subject}: wrong RTGP commit")
    require(source_hash == RTGP_ARCHIVE_SHA256, f"{subject}: wrong RTGP archive")
    require(pair_names.shape == (28,), f"{subject}: wrong pair-name shape")

    train_labels = y_train.reshape(-1, 6)
    validation_labels = y_validation.reshape(-1, 6)
    require(np.all(train_labels.sum(axis=1) == 1),
            f"{subject}: invalid train six-choice labels")
    require(np.all(validation_labels.sum(axis=1) == 1),
            f"{subject}: invalid validation six-choice labels")

    return {
        "train_eeg": x_train.reshape(-1, 6, 8, 128, 1),
        "train_interactions": x0_train.reshape(-1, 6, 28),
        "train_labels": train_labels.astype(np.float32),
        "validation_eeg": x_validation.reshape(-1, 6, 8, 128, 1),
        "validation_interactions": x0_validation.reshape(-1, 6, 28),
        "validation_labels": validation_labels.astype(np.float32),
        "pair_names": pair_names,
    }


def build_epoch_scorer(EEG_DBNet_V2):
    binary_model = EEG_DBNet_V2(
        NumFilter=8,
        SamplingFrequency=128,
        NumChannels=8,
        FilterScaler=2,
        NumClasses=2,
        DropoutRate=0.5,
    ).build_model()
    require(isinstance(binary_model.layers[-3], tf.keras.layers.Concatenate),
            "Certified DBNet layer layout changed")
    feature_extractor = tf.keras.Model(
        binary_model.input,
        binary_model.layers[-3].output,
        name="dbnet_v2_feature_extractor",
    )
    epoch_input = tf.keras.Input(shape=(8, 128, 1), name="single_stimulus_epoch")
    score = tf.keras.layers.Dense(
        1,
        kernel_constraint=tf.keras.constraints.max_norm(0.25),
        name="stimulus_score",
    )(feature_extractor(epoch_input))
    return tf.keras.Model(epoch_input, score, name="dbnet_v2_epoch_scorer")


def build_interaction_dbnet(EEG_DBNet_V2):
    epoch_scorer = build_epoch_scorer(EEG_DBNet_V2)
    eeg_input = tf.keras.Input(shape=(6, 8, 128, 1), name="six_stimulus_eeg")
    interaction_input = tf.keras.Input(
        shape=(6, 28), name="six_stimulus_rtgp_interactions"
    )
    dbnet_scores = tf.keras.layers.TimeDistributed(
        epoch_scorer, name="shared_dbnet_scorer"
    )(eeg_input)
    dbnet_logits = tf.keras.layers.Reshape((6,), name="dbnet_logits")(dbnet_scores)
    interaction_scores = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(
            1,
            use_bias=False,
            kernel_initializer="zeros",
            name="interaction_linear",
        ),
        name="shared_rtgp_interaction_scorer",
    )(interaction_input)
    interaction_logits = tf.keras.layers.Reshape(
        (6,), name="interaction_logits_unscaled"
    )(interaction_scores)
    interaction_logits = tf.keras.layers.Rescaling(
        1.0 / 28.0, name="rtgp_ns2_scaling"
    )(interaction_logits)
    combined_logits = tf.keras.layers.Add(name="combined_choice_logits")(
        [dbnet_logits, interaction_logits]
    )
    probabilities = tf.keras.layers.Softmax(
        axis=-1, name="six_choice_probabilities"
    )(combined_logits)
    model = tf.keras.Model(
        [eeg_input, interaction_input],
        probabilities,
        name="structured_dbnet_plus_rtgp_interactions",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[tf.keras.metrics.CategoricalAccuracy(name="six_choice_accuracy")],
    )
    require(model.count_params() == 3989,
            f"Expected 3989 parameters, found {model.count_params()}")
    return model


def interaction_kernel(model):
    weights = model.get_layer("shared_rtgp_interaction_scorer").layer.get_weights()
    require(len(weights) == 1 and weights[0].shape == (28, 1),
            "Unexpected interaction branch weights")
    return np.asarray(weights[0]).reshape(28)


def structured_validation_metrics(probabilities, labels):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.float64)
    require(probabilities.shape == (140, 6),
            f"Expected probabilities (140,6), found {probabilities.shape}")
    require(labels.shape == (140, 6),
            f"Expected labels (140,6), found {labels.shape}")
    require(np.isfinite(probabilities).all(), "Non-finite validation probabilities")
    np.testing.assert_allclose(
        probabilities.sum(axis=1), 1.0, rtol=1e-5, atol=1e-5
    )
    grouped_p = probabilities.reshape(7, 10, 2, 6)
    grouped_y = labels.reshape(7, 10, 2, 6)
    cumulative = np.cumsum(grouped_p, axis=1)
    predicted = cumulative.argmax(axis=-1)
    truth = grouped_y.argmax(axis=-1)
    character_curve = np.all(predicted == truth, axis=2).mean(axis=0)
    clipped = np.clip(probabilities, 1e-8, 1.0)
    nll = float(-np.mean(np.sum(labels * np.log(clipped), axis=1)))
    return {
        "structured_nll": nll,
        "half_sequence_accuracy": float(
            np.mean(probabilities.argmax(axis=1) == labels.argmax(axis=1))
        ),
        "character_accuracy_4": float(character_curve[3]),
        "character_accuracy_7": float(character_curve[6]),
        "character_accuracy_10": float(character_curve[9]),
        "character_accuracy_curve": character_curve.tolist(),
    }


def load_control_metrics(control_root, subject):
    path = control_root / subject / "seed_1" / "validation_metrics.json"
    require(path.is_file(), f"Missing matched structured-DBNet control: {path}")
    with path.open("r", encoding="utf-8") as handle:
        record = json.load(handle)
    require(record.get("subject") == subject, f"Wrong control subject in {path}")
    require(int(record.get("classifier_seed")) == 1, f"Wrong control seed in {path}")
    require(int(record.get("parameters")) == 3961,
            f"Wrong control parameter count in {path}")
    require(record.get("test_data_accessed") is False,
            f"Control metrics do not certify validation-only use: {path}")
    for key in (
        "structured_nll", "half_sequence_accuracy", "character_accuracy_4",
        "character_accuracy_7", "character_accuracy_10",
    ):
        require(key in record and np.isfinite(float(record[key])),
                f"Control metric {key} missing/non-finite in {path}")
    return record


def verified_completed_run(run_dir, subject):
    required = [
        run_dir / "completed.json",
        run_dir / "validation_metrics.json",
        run_dir / "validation_predictions.npz",
        run_dir / "interaction_coefficients.npz",
        run_dir / "best.weights.h5",
        run_dir / "history.csv",
    ]
    if not all(path.is_file() for path in required):
        return None
    try:
        with (run_dir / "completed.json").open("r", encoding="utf-8") as handle:
            completed = json.load(handle)
        with (run_dir / "validation_metrics.json").open(
            "r", encoding="utf-8"
        ) as handle:
            metrics = json.load(handle)
        if (
            completed.get("status") != "completed"
            or completed.get("protocol_id") != MODEL_PROTOCOL_ID
            or metrics.get("protocol_id") != MODEL_PROTOCOL_ID
            or metrics.get("subject") != subject
            or int(metrics.get("classifier_seed")) != 1
            or int(metrics.get("parameters")) != 3989
            or metrics.get("test_data_accessed") is not False
        ):
            return None
        with np.load(run_dir / "validation_predictions.npz", allow_pickle=False) as archive:
            probabilities = np.asarray(archive["probabilities"])
            labels = np.asarray(archive["labels"])
        with np.load(run_dir / "interaction_coefficients.npz", allow_pickle=False) as archive:
            coefficients = np.asarray(archive["coefficients"])
            pair_names = np.asarray(archive["pair_names"])
        if probabilities.shape != (140, 6) or labels.shape != (140, 6):
            return None
        if coefficients.shape != (28,) or pair_names.shape != (28,):
            return None
        recalculated = structured_validation_metrics(probabilities, labels)
        for key in (
            "structured_nll", "half_sequence_accuracy", "character_accuracy_4",
            "character_accuracy_7", "character_accuracy_10",
        ):
            if not np.isclose(float(metrics[key]), recalculated[key], atol=1e-7):
                return None
        return metrics
    except (OSError, ValueError, KeyError, TypeError, AssertionError, json.JSONDecodeError):
        return None


class CompactProgress(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        number = epoch + 1
        if number == 1 or number % 25 == 0:
            coefficients = interaction_kernel(self.model)
            print(
                f"epoch {number:4d} | "
                f"loss {logs.get('loss', float('nan')):.4f} | "
                f"val_loss {logs.get('val_loss', float('nan')):.4f} | "
                f"val_accuracy "
                f"{logs.get('val_six_choice_accuracy', float('nan')):.4f} | "
                f"interaction_norm {np.linalg.norm(coefficients):.4f}",
                flush=True,
            )


def main():
    project = resolve_project()
    prepared_dir = project / "GLASS_GANet/prepared_data"
    experiment_root = (
        project / "GLASS_GANet/results/multisubject_replication/"
        "dbnet_interaction_pilot"
    )
    feature_dir = experiment_root / "interaction_features"
    training_root = experiment_root / "interaction_dbnet_training"
    control_root = (
        project / "GLASS_GANet/results/multisubject_replication/"
        "dbnet_glass_structured_pilot/structured_dbnet_training"
    )
    training_root.mkdir(parents=True, exist_ok=True)

    gpu_devices = tf.config.list_physical_devices("GPU")
    if not gpu_devices:
        raise RuntimeError(
            "No GPU is visible. In Colab select Runtime > Change runtime type > "
            "T4 GPU, reconnect, mount Drive, and rerun this script."
        )

    step2_path = latest_step2_protocol(experiment_root)
    EEG_DBNet_V2, dbnet_source = load_public_dbnet(project)
    control_records = {
        subject: load_control_metrics(control_root, subject) for subject in SUBJECTS
    }

    print("Project:", project)
    print("Training output:", training_root)
    print("DBNet source:", dbnet_source)
    print("Frozen Step-2 protocol:", step2_path.name)
    print("TensorFlow:", tf.__version__)
    print("GPU:", gpu_devices)
    print("Subjects:", SUBJECTS)
    print("Classifier seed:", TRAINING_SEED)
    print("Maximum epochs:", MAXIMUM_EPOCHS)
    print("Patience:", PATIENCE)
    print("Training/validation only; test values are not loaded.\n")

    results = []
    for subject in SUBJECTS:
        print("\n" + "=" * 72)
        print(f"{subject} | DBNet + RTGP interactions | seed {TRAINING_SEED}")
        print("=" * 72)
        run_dir = training_root / subject / "seed_1"
        run_dir.mkdir(parents=True, exist_ok=True)

        completed = verified_completed_run(run_dir, subject)
        if completed is not None:
            print("Skipping verified completed run.")
            results.append(completed)
            continue
        if any(run_dir.iterdir()):
            print("Incomplete or incompatible run detected; restarting this subject.")

        data = load_grouped_subject(prepared_dir, feature_dir, subject)
        require(data["train_eeg"].shape == (420, 6, 8, 128, 1),
                f"{subject}: wrong grouped training EEG")
        require(data["validation_eeg"].shape == (140, 6, 8, 128, 1),
                f"{subject}: wrong grouped validation EEG")

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(TRAINING_SEED)
        model = build_interaction_dbnet(EEG_DBNet_V2)
        initial_kernel = interaction_kernel(model)
        require(np.count_nonzero(initial_kernel) == 0,
                f"{subject}: interaction branch did not start at zero")

        best_weights = run_dir / "best.weights.h5"
        history_path = run_dir / "history.csv"
        callbacks = [
            tf.keras.callbacks.TerminateOnNaN(),
            tf.keras.callbacks.ModelCheckpoint(
                filepath=str(best_weights),
                monitor="val_loss",
                mode="min",
                save_best_only=True,
                save_weights_only=True,
                verbose=0,
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                mode="min",
                patience=PATIENCE,
                min_delta=MINIMUM_DELTA,
                restore_best_weights=True,
                verbose=1,
            ),
            tf.keras.callbacks.CSVLogger(str(history_path), append=False),
            CompactProgress(),
        ]

        start = time.perf_counter()
        history = model.fit(
            [data["train_eeg"], data["train_interactions"]],
            data["train_labels"],
            validation_data=(
                [data["validation_eeg"], data["validation_interactions"]],
                data["validation_labels"],
            ),
            epochs=MAXIMUM_EPOCHS,
            batch_size=GROUP_BATCH_SIZE,
            shuffle=True,
            callbacks=callbacks,
            verbose=0,
        )
        training_seconds = float(time.perf_counter() - start)

        require(best_weights.is_file(), f"Best weights missing for {subject}")
        model.load_weights(best_weights)
        probabilities = model.predict(
            [data["validation_eeg"], data["validation_interactions"]],
            batch_size=GROUP_BATCH_SIZE,
            verbose=0,
        ).astype(np.float32)
        metrics = structured_validation_metrics(
            probabilities, data["validation_labels"]
        )
        validation_losses = np.asarray(history.history["val_loss"], dtype=np.float64)
        best_epoch = int(validation_losses.argmin() + 1)
        epochs_trained = int(len(validation_losses))
        coefficients = interaction_kernel(model).astype(np.float32)
        ranking = np.argsort(np.abs(coefficients))[::-1]
        top_interactions = [
            {
                "pair": str(data["pair_names"][index]),
                "coefficient": float(coefficients[index]),
            }
            for index in ranking[:5]
        ]

        control = control_records[subject]
        deltas = {
            key: float(metrics[key] - float(control[key]))
            for key in (
                "structured_nll", "half_sequence_accuracy", "character_accuracy_4",
                "character_accuracy_7", "character_accuracy_10",
            )
        }
        result = {
            "subject": subject,
            "classifier_seed": TRAINING_SEED,
            "configuration": "structured_dbnet_plus_rtgp_interactions",
            "protocol_id": MODEL_PROTOCOL_ID,
            "parameters": int(model.count_params()),
            "added_parameters_vs_control": 28,
            "best_epoch": best_epoch,
            "epochs_trained": epochs_trained,
            "training_seconds": training_seconds,
            "training_minutes": training_seconds / 60.0,
            "interaction_coefficient_l2_norm": float(np.linalg.norm(coefficients)),
            "interaction_coefficient_max_abs": float(np.max(np.abs(coefficients))),
            "top_interactions": top_interactions,
            **metrics,
            "delta_vs_structured_dbnet_seed1": deltas,
            "test_data_accessed": False,
        }

        save_npz_safely(
            run_dir / "validation_predictions.npz",
            probabilities=probabilities,
            labels=data["validation_labels"].astype(np.int8),
            best_epoch=np.int32(best_epoch),
            protocol_id=np.array(MODEL_PROTOCOL_ID),
        )
        save_npz_safely(
            run_dir / "interaction_coefficients.npz",
            coefficients=coefficients,
            pair_names=data["pair_names"],
            protocol_id=np.array(MODEL_PROTOCOL_ID),
        )
        write_json_safely(result, run_dir / "validation_metrics.json")
        write_json_safely(
            {
                "status": "completed",
                "subject": subject,
                "classifier_seed": TRAINING_SEED,
                "protocol_id": MODEL_PROTOCOL_ID,
                "completed_utc": datetime.now(timezone.utc).isoformat(),
                "test_data_accessed": False,
            },
            run_dir / "completed.json",
        )
        results.append(result)
        print(
            f"Completed | best epoch {best_epoch} | "
            f"minutes {training_seconds / 60.0:.2f} | "
            f"NLL {metrics['structured_nll']:.4f} | "
            f"4-seq {metrics['character_accuracy_4']:.4f} | "
            f"7-seq {metrics['character_accuracy_7']:.4f} | "
            f"10-seq {metrics['character_accuracy_10']:.4f}"
        )
        print("Largest learned interaction coefficients:", top_interactions[:3])

    require(len(results) == 8, f"Expected 8 completed runs, found {len(results)}")
    results_frame = pd.DataFrame(results).sort_values("subject")

    metric_keys = [
        "structured_nll", "half_sequence_accuracy", "character_accuracy_4",
        "character_accuracy_7", "character_accuracy_10",
    ]
    comparison_rows = []
    for subject in SUBJECTS:
        candidate = next(item for item in results if item["subject"] == subject)
        control = control_records[subject]
        row = {"subject": subject}
        for key in metric_keys:
            row[f"control_{key}"] = float(control[key])
            row[f"candidate_{key}"] = float(candidate[key])
            row[f"delta_{key}"] = float(candidate[key]) - float(control[key])
        comparison_rows.append(row)
    comparison_frame = pd.DataFrame(comparison_rows).sort_values("subject")

    control_means = {
        key: float(np.mean([float(control_records[s][key]) for s in SUBJECTS]))
        for key in metric_keys
    }
    candidate_means = {
        key: float(results_frame[key].astype(float).mean()) for key in metric_keys
    }
    mean_deltas = {
        key: candidate_means[key] - control_means[key] for key in metric_keys
    }
    primary_improved = mean_deltas["character_accuracy_4"] > 0
    directional_support = (
        primary_improved
        and mean_deltas["structured_nll"] <= 0
        and mean_deltas["half_sequence_accuracy"] >= 0
    )

    summary = {
        "step": "interaction_step3a",
        "status": "completed",
        "protocol_id": MODEL_PROTOCOL_ID,
        "protocol": MODEL_PROTOCOL,
        "subjects": SUBJECTS,
        "completed_runs": 8,
        "control": "structured_dbnet_seed1",
        "candidate": "structured_dbnet_plus_rtgp_interactions_seed1",
        "control_mean": control_means,
        "candidate_mean": candidate_means,
        "candidate_minus_control": mean_deltas,
        "primary_4_sequence_metric_improved": bool(primary_improved),
        "directionally_supportive_screen": bool(directional_support),
        "interpretation": (
            "Validation-only exploratory screen. It cannot establish final test "
            "superiority, rule out selection bias, or justify a publication claim."
        ),
        "total_training_minutes": float(results_frame["training_minutes"].sum()),
        "test_data_accessed": False,
    }

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    results_path = training_root / f"seed1_validation_results_{timestamp}.csv"
    comparison_path = training_root / f"seed1_matched_comparison_{timestamp}.csv"
    summary_path = training_root / f"seed1_validation_summary_{timestamp}.json"
    results_frame.drop(columns=["top_interactions", "character_accuracy_curve",
                                "delta_vs_structured_dbnet_seed1"]).to_csv(
        results_path, index=False
    )
    comparison_frame.to_csv(comparison_path, index=False)
    write_json_safely(summary, summary_path)

    print("\nSeed-1 matched validation comparison")
    print("Negative delta is better only for NLL; positive is better for accuracies.")
    print(f"{'Metric':34s} {'Control':>10s} {'Candidate':>11s} {'Delta':>10s}")
    for key in metric_keys:
        print(
            f"{key:34s} {control_means[key]:10.4f} "
            f"{candidate_means[key]:11.4f} {mean_deltas[key]:10.4f}"
        )
    print("\nPrimary 4-sequence metric improved:", primary_improved)
    print("Directionally supportive validation screen:", directional_support)
    print("Total training minutes:", round(summary["total_training_minutes"], 2))
    print("\nSaved:")
    print(results_path)
    print(comparison_path)
    print(summary_path)
    print("\nSTEP 3A COMPLETED")
    print("Interaction DBNet seed 1 completed for A01-A08.")
    print("No test EEG, labels, predictions, or metrics were loaded.")
    print("Share the full comparison table before running seeds 2 and 3.")


if __name__ == "__main__":
    main()


In [ ]:
from google.colab import drive, files

drive.mount("/content/drive")